# Coleta de Dados: Acidentes de Trânsito em SP

**Fonte principal:** [Infosiga SP](https://www.infosiga.sp.gov.br) — baixe o ZIP em Dados Abertos e faça upload abaixo  
**Fonte secundária:** IBGE — Malha municipal do estado de SP (download automático)  

O ZIP do Infosiga contém 6 arquivos. Usamos apenas os **sinistros** (que têm lat/lon e gravidade):
- `sinistros_2015-2021.csv`
- `sinistros_2022-2026.csv`

**Referências:**  
INFOSIGA SP. *Dados Abertos*. Disponível em: https://www.infosiga.sp.gov.br  
IBGE. *Malha Municipal 2022*. Disponível em: https://geoftp.ibge.gov.br

## 1. Instalação de dependências

In [1]:
!pip install geopandas folium psycopg2-binary sqlalchemy GeoAlchemy2 mapclassify --quiet
print('Dependências instaladas')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 3.1 MB/s eta 0:00:00
Dependências instaladas


## 2. Imports

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import json, os, io, zipfile, unicodedata, warnings
import urllib.request
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

print('Imports OK')
print(f'GeoPandas: {gpd.__version__} | Pandas: {pd.__version__}')

Imports OK
GeoPandas: 1.1.3 | Pandas: 2.2.2


## 3. Configuração do Banco de Dados



In [3]:
DB_HOST = 'localhost'
DB_PORT = 5432
DB_NAME = 'acidentes_sp'
DB_USER = 'postgres'
DB_PASS = 'postgres'

DB_URL = f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

DATA_DIR = '/content/dados'
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Banco: {DB_NAME} em {DB_HOST}:{DB_PORT}')
print(f'Dados: {DATA_DIR}')

Banco: acidentes_sp em localhost:5432
Dados: /content/dados


## 4. Subir PostgreSQL + PostGIS no Colab

> Execute esta célula apenas uma vez por sessão.

In [4]:
!sudo apt-get install -y postgresql postgis > /dev/null 2>&1
!sudo service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres psql -c "CREATE DATABASE acidentes_sp;"
!sudo -u postgres psql -d acidentes_sp -c "CREATE EXTENSION postgis;"
print('PostgreSQL + PostGIS prontos!')

 * Starting PostgreSQL 14 database server
   ...done.
ALTER ROLE
CREATE DATABASE
CREATE EXTENSION
PostgreSQL + PostGIS prontos!


## 5. Criação do Schema PostGIS

In [5]:
SCHEMA_SQL = '''
DROP TABLE IF EXISTS acidentes CASCADE;
DROP TABLE IF EXISTS municipios CASCADE;

CREATE TABLE municipios (
    id          SERIAL PRIMARY KEY,
    codigo_ibge TEXT UNIQUE NOT NULL,
    nome        TEXT NOT NULL,
    geom        GEOMETRY(MultiPolygon, 4326)
);
CREATE INDEX idx_municipios_geom ON municipios USING GIST (geom);

CREATE TABLE acidentes (
    id              SERIAL PRIMARY KEY,
    id_sinistro     TEXT,
    ano             INT,
    mes             INT,
    data_ocorrencia DATE,
    dia_semana      TEXT,
    hora            TEXT,
    turno           TEXT,
    tipo_local      TEXT,
    municipio_id    INT REFERENCES municipios(id) ON DELETE SET NULL,
    municipio_nome  TEXT,
    regiao_administrativa TEXT,
    tipo_acidente   TEXT,
    causa_acidente  TEXT,
    mortos          INT DEFAULT 0,
    feridos_graves  INT DEFAULT 0,
    feridos_leves   INT DEFAULT 0,
    ilesos          INT DEFAULT 0,
    total_vitimas   INT DEFAULT 0,
    latitude        DOUBLE PRECISION,
    longitude       DOUBLE PRECISION,
    geom            GEOMETRY(Point, 4326),
    fonte           TEXT DEFAULT \'Infosiga SP\'
);
CREATE INDEX idx_acidentes_geom      ON acidentes USING GIST (geom);
CREATE INDEX idx_acidentes_municipio ON acidentes (municipio_id);
CREATE INDEX idx_acidentes_data      ON acidentes (data_ocorrencia);
CREATE INDEX idx_acidentes_ano_mes   ON acidentes (ano, mes);
'''

try:
    engine = create_engine(DB_URL)
    with engine.connect() as conn:
        for stmt in SCHEMA_SQL.split(';'):
            stmt = stmt.strip()
            if stmt:
                conn.execute(text(stmt))
        conn.commit()
    print('Schema criado com sucesso!')
except Exception as e:
    print(f'Erro: {e}')

Schema criado com sucesso!


## 6. Download da Malha Municipal — IBGE

In [6]:
IBGE_URL = (
    'https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/'
    'malhas_municipais/municipio_2022/UFs/SP/SP_Municipios_2022.zip'
)
zip_path = os.path.join(DATA_DIR, 'SP_Municipios_2022.zip')
shp_dir  = os.path.join(DATA_DIR, 'municipios_sp')

print('Baixando malha IBGE...')
urllib.request.urlretrieve(IBGE_URL, zip_path)
os.makedirs(shp_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(shp_dir)
print(f'Download OK')

shp_file = [f for f in os.listdir(shp_dir) if f.endswith('.shp')][0]
gdf_municipios = gpd.read_file(os.path.join(shp_dir, shp_file)).to_crs(epsg=4326)
gdf_municipios = gdf_municipios.rename(columns={'CD_MUN': 'codigo_ibge', 'NM_MUN': 'nome'})[['codigo_ibge','nome','geometry']]
print(f'{len(gdf_municipios)} municípios carregados | CRS: {gdf_municipios.crs}')
gdf_municipios.head(3)

⏳ Baixando malha IBGE...
Download OK
645 municípios carregados | CRS: EPSG:4326


,codigo_ibge,nome,geometry
0,3500105,Adamantina,"POLYGON ((-51.09557 -21.57029, -51.09617 -21.5..."
1,3500204,Adolfo,"POLYGON ((-49.61249 -21.2611, -49.61249 -21.26..."
2,3500303,Aguaí,"POLYGON ((-47.01254 -22.00527, -47.01219 -22.0..."


## 7. Upload do ZIP do Infosiga SP

> Baixe o ZIP em https://www.infosiga.sp.gov.br → Dados Abertos, depois faça upload aqui.

In [7]:
from google.colab import files

print('Selecione o arquivo ZIP baixado do Infosiga SP...')
uploaded = files.upload()

dfs_sinistros = []
for fname, content in uploaded.items():
    with zipfile.ZipFile(io.BytesIO(content)) as zf:
        print(f'Arquivos no ZIP: {zf.namelist()}')
        for nome_arq in zf.namelist():
            # Usar apenas os CSVs de sinistros
            if 'sinistro' in nome_arq.lower() and nome_arq.endswith('.csv'):
                with zf.open(nome_arq) as f:
                    df_tmp = pd.read_csv(f, sep=';', encoding='latin-1')
                    dfs_sinistros.append(df_tmp)
                    print(f'{nome_arq}: {len(df_tmp):,} registros')
            else:
                print(f'{nome_arq} ignorado (não é sinistro)')

df_raw = pd.concat(dfs_sinistros, ignore_index=True)
print(f'\nTotal bruto: {len(df_raw):,} sinistros')
print(f'    Colunas: {df_raw.columns.tolist()}')

Selecione o arquivo ZIP baixado do Infosiga SP...


Saving dados_infosiga.zip to dados_infosiga.zip
Arquivos no ZIP: ['pessoas_2015-2021.csv', 'pessoas_2022-2026.csv', 'sinistros_2015-2021.csv', 'sinistros_2022-2026.csv', 'veiculos_2015-2021.csv', 'veiculos_2022-2026.csv']
pessoas_2015-2021.csv ignorado (não é sinistro)
pessoas_2022-2026.csv ignorado (não é sinistro)
sinistros_2015-2021.csv: 535,566 registros
sinistros_2022-2026.csv: 814,085 registros
veiculos_2015-2021.csv ignorado (não é sinistro)
veiculos_2022-2026.csv ignorado (não é sinistro)

Total bruto: 1,349,651 sinistros
    Colunas: ['id_sinistro', 'tipo_registro', 'data_sinistro', 'ano_sinistro', 'mes_sinistro', 'dia_sinistro', 'hora_sinistro', 'ano_mes_sinistro', 'dia_da_semana', 'turno', 'logradouro', 'numero_logradouro', 'tipo_via', 'tipo_local', 'latitude', 'longitude', 'cod_ibge', 'municipio', 'regiao_administrativa', 'administracao', 'conservacao', 'circunscricao', 'tp_sinistro_primario', 'qtd_pedestre', 'qtd_bicicleta', 'qtd_motocicleta', 'qtd_automovel', 'qtd_onibus'

## 8. Limpeza e Padronização

In [8]:
# Filtrar e renomear colunas dos sinistros para o nosso schema
df_sin = df_raw[df_raw.columns[df_raw.columns.isin([
    'id_sinistro','data_sinistro','ano_sinistro','mes_sinistro','dia_sinistro',
    'hora_sinistro','dia_da_semana','turno','tipo_via','tipo_local',
    'latitude','longitude','cod_ibge','municipio','regiao_administrativa',
    'tp_sinistro_primario','qtd_gravidade_fatal','qtd_gravidade_grave',
    'qtd_gravidade_leve','qtd_gravidade_ileso'
])]].dropna(subset=['hora_sinistro']).copy()

df = df_sin.rename(columns={
    'id_sinistro'         : 'id_sinistro',
    'ano_sinistro'        : 'ano',
    'mes_sinistro'        : 'mes',
    'data_sinistro'       : 'data_ocorrencia',
    'dia_da_semana'       : 'dia_semana',
    'hora_sinistro'       : 'hora',
    'municipio'           : 'municipio_nome',
    'tp_sinistro_primario': 'tipo_acidente',
    'tipo_via'            : 'causa_acidente',
    'qtd_gravidade_fatal' : 'mortos',
    'qtd_gravidade_grave' : 'feridos_graves',
    'qtd_gravidade_leve'  : 'feridos_leves',
    'qtd_gravidade_ileso' : 'ilesos',
    'cod_ibge'            : 'codigo_ibge',
}).copy()

# Numéricos
for col in ['mortos','feridos_graves','feridos_leves','ilesos']:
    if col not in df.columns: df[col] = 0
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
df['total_vitimas'] = df['mortos'] + df['feridos_graves'] + df['feridos_leves']

# Data
df['data_ocorrencia'] = pd.to_datetime(df['data_ocorrencia'], errors='coerce', dayfirst=True)
if 'ano' not in df.columns: df['ano'] = df['data_ocorrencia'].dt.year
if 'mes' not in df.columns: df['mes'] = df['data_ocorrencia'].dt.month

# Texto
for col in ['municipio_nome','tipo_acidente','causa_acidente','dia_semana','turno','tipo_local','regiao_administrativa']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()

# Coordenadas
for col in ['latitude','longitude']:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',','.'), errors='coerce')

# Filtrar SP
df = df[df['latitude'].between(-26,-19) & df['longitude'].between(-54,-44)]

print(f'Limpeza OK: {len(df):,} sinistros')
print(f'   Período: {df["ano"].min()} – {df["ano"].max()}')
print(f'   Com coordenadas: {df[["latitude","longitude"]].dropna().shape[0]:,}')
df.head(3)

Limpeza OK: 1,243,731 sinistros
   Período: 2014 – 2026
   Com coordenadas: 1,243,731


,id_sinistro,data_ocorrencia,ano,mes,dia_sinistro,hora,dia_semana,turno,causa_acidente,tipo_local,...,longitude,codigo_ibge,municipio_nome,regiao_administrativa,tipo_acidente,mortos,feridos_graves,feridos_leves,ilesos,total_vitimas
0,2501575,2014-12-21,2014,12,21,20:00,DOMINGO,NOITE,VIAS URBANAS,PUBLICO,...,-47.182293,3509502,CAMPINAS,CAMPINAS,ATROPELAMENTO,1,0,0,0,1
2,2463759,2014-12-26,2014,12,26,06:52,SEXTA-FEIRA,MANHA,VIAS URBANAS,PUBLICO,...,-46.666539,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,ATROPELAMENTO,1,0,0,0,1
3,2487781,2014-12-28,2014,12,28,14:30,DOMINGO,TARDE,VIAS URBANAS,PUBLICO,...,-46.849214,3510609,CARAPICUIBA,METROPOLITANA DE SÃO PAULO,ATROPELAMENTO,1,0,0,0,1


## 9. Join com Municípios IBGE

In [9]:
def normalizar(t):
    if pd.isna(t): return ''
    t = unicodedata.normalize('NFKD', str(t).upper().strip())
    return ''.join(c for c in t if not unicodedata.combining(c))

gdf_municipios['nome_norm'] = gdf_municipios['nome'].apply(normalizar)
df['municipio_norm']        = df['municipio_nome'].apply(normalizar)

# Tentar match por nome; se tiver cod_ibge no df, usar diretamente
if 'codigo_ibge' in df.columns:
    # cod_ibge do Infosiga tem 6 dígitos, IBGE tem 7 — completar com 0 no final se necessário
    df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.strip()
    gdf_municipios['codigo_ibge6'] = gdf_municipios['codigo_ibge'].str[:6]
    ibge6_to_id_full = dict(zip(gdf_municipios['codigo_ibge6'], gdf_municipios['codigo_ibge']))
    df['codigo_ibge'] = df['codigo_ibge'].map(ibge6_to_id_full).fillna(df['codigo_ibge'])

nome_to_ibge = dict(zip(gdf_municipios['nome_norm'], gdf_municipios['codigo_ibge']))
df['codigo_ibge'] = df['codigo_ibge'].fillna(df['municipio_norm'].map(nome_to_ibge))

pct = df['codigo_ibge'].notna().mean() * 100
print(f'Match municípios: {pct:.1f}%')

Match municípios: 100.0%


## 10. Inserção no PostgreSQL

In [16]:
# Limpar tabelas antes de reinserir
with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS acidentes CASCADE'))
    conn.execute(text('DROP TABLE IF EXISTS municipios CASCADE'))
    conn.commit()
print('Tabelas limpas, pode inserir!')

# Inserindo municípios
print('Inserindo municípios...')

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS acidentes CASCADE'))
    conn.execute(text('DROP TABLE IF EXISTS municipios CASCADE'))
    conn.execute(text('''
        CREATE TABLE municipios (
            id          SERIAL PRIMARY KEY,
            codigo_ibge TEXT UNIQUE NOT NULL,
            nome        TEXT NOT NULL,
            geom        GEOMETRY(MultiPolygon, 4326)
        )
    '''))
    conn.commit()

# Inserir usando WKT
with engine.connect() as conn:
    for _, row in gdf_municipios.iterrows():
        conn.execute(text('''
            INSERT INTO municipios (codigo_ibge, nome, geom)
            VALUES (:codigo_ibge, :nome, ST_GeomFromText(:wkt, 4326))
            ON CONFLICT (codigo_ibge) DO NOTHING
        '''), {
            'codigo_ibge': row['codigo_ibge'],
            'nome': row['nome'],
            'wkt': row['geometry'].wkt
        })
    conn.commit()

print(f'{len(gdf_municipios)} municípios inseridos')

# Buscar IDs
with engine.connect() as conn:
    mun_db = pd.read_sql('SELECT id, codigo_ibge FROM municipios', conn)
df['municipio_id'] = df['codigo_ibge'].map(dict(zip(mun_db['codigo_ibge'], mun_db['id'])))

# Selecionar colunas do schema
cols = ['id_sinistro','ano','mes','data_ocorrencia','dia_semana','hora','turno',
        'tipo_local','municipio_id','municipio_nome','regiao_administrativa',
        'tipo_acidente','causa_acidente','mortos','feridos_graves','feridos_leves',
        'ilesos','total_vitimas','latitude','longitude']
df_insert = df[[c for c in cols if c in df.columns]].copy()

print(f'Inserindo {len(df_insert):,} sinistros...')
df_insert.to_sql('acidentes', engine, if_exists='append', index=False, chunksize=10000, method='multi')
print('Sinistros inseridos!')

# Adicionar coluna geom e atualizar
with engine.connect() as conn:
    conn.execute(text('ALTER TABLE acidentes ADD COLUMN IF NOT EXISTS geom GEOMETRY(Point, 4326)'))
    conn.execute(text('''
        UPDATE acidentes SET geom = ST_SetSRID(ST_MakePoint(longitude, latitude), 4326)
        WHERE latitude IS NOT NULL AND longitude IS NOT NULL
          AND latitude BETWEEN -90 AND 90 AND longitude BETWEEN -180 AND 180
    '''))
    conn.execute(text('CREATE INDEX IF NOT EXISTS idx_acidentes_geom ON acidentes USING GIST (geom)'))
    conn.commit()
print('Coluna geom adicionada e geometrias atualizadas!')

Tabelas limpas, pode inserir!
Inserindo municípios...
645 municípios inseridos
Inserindo 1,243,731 sinistros...
Sinistros inseridos!
Coluna geom adicionada e geometrias atualizadas!


## 11. Exportação para JSON (backup)

In [17]:
# GeoJSON municípios
gdf_municipios.to_file(os.path.join(DATA_DIR,'municipios_sp.geojson'), driver='GeoJSON')
print('municipios_sp.geojson salvo')

# JSON sinistros
df_json = df_insert.copy()
if 'data_ocorrencia' in df_json.columns:
    df_json['data_ocorrencia'] = df_json['data_ocorrencia'].astype(str)
json_path = os.path.join(DATA_DIR, 'sinistros_sp.json')
df_json.to_json(json_path, orient='records', force_ascii=False, indent=2)
print(f'sinistros_sp.json salvo ({os.path.getsize(json_path)/1024/1024:.1f} MB)')

municipios_sp.geojson salvo
sinistros_sp.json salvo (644.2 MB)


## 12. Verificação Final

In [20]:
with engine.connect() as conn:
    n_mun   = conn.execute(text('SELECT COUNT(*) FROM municipios')).scalar()
    n_acid  = conn.execute(text('SELECT COUNT(*) FROM acidentes')).scalar()
    n_geom  = conn.execute(text('SELECT COUNT(*) FROM acidentes WHERE geom IS NOT NULL')).scalar()
    periodo = conn.execute(text('SELECT MIN(ano), MAX(ano) FROM acidentes')).fetchone()
    mortos  = conn.execute(text('SELECT SUM(mortos) FROM acidentes')).scalar()

print('=' * 55)
print('RESUMO DOS DADOS COLETADOS')
print('=' * 55)
print(f'  Municípios cadastrados  : {n_mun:>8,}')
print(f'  Total de sinistros      : {n_acid:>8,}')
print(f'  Com geolocalização      : {n_geom:>8,}')
print(f'  Período coberto         :    {periodo[0]} – {periodo[1]}')
print(f'  Total de mortos         : {mortos:>8,}')
print('=' * 55)
print('\nColeta e armazenamento concluídos!')

# Salvar backup do banco no Google Drive (rodar uma vez após a coleta)
from google.colab import drive
drive.mount('/content/drive')

!sudo -u postgres pg_dump acidentes_sp > /content/drive/MyDrive/acidentes_sp_backup.sql
print('Backup salvo no Google Drive!')

RESUMO DOS DADOS COLETADOS
  Municípios cadastrados  :      645
  Total de sinistros      : 1,243,731
  Com geolocalização      : 1,243,731
  Período coberto         :    2014 – 2026
  Total de mortos         :   52,283

Coleta e armazenamento concluídos!
Mounted at /content/drive
Backup salvo no Google Drive!
